In [0]:
from pyspark.sql.functions import to_timestamp, current_timestamp, monotonically_increasing_id, lit, col

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import TimestampType, DateType
from delta.tables import DeltaTable

In [0]:
catalog = 'labuser11612924_1758377596'
bronze_tbl = f"{catalog}.bronze.customers"

In [0]:
CATALOG = "labuser11612924_1758377596"
SCHEMA  = "silver"
DIM_TBL = "dim_customers"

BRONZE_CATALOG = "labuser11612924_1758377596"
BRONZE_SCHEMA  = "bronze"
BRONZE_TBL     = "customers"  # bronze source table name (append-only)

PK_COL = "customer_id"   # natural/business key

# Columns that define SCD2 changes (Type 2 tracked fields)
SCD2_COLUMNS = ["email","city","state"]

# Columns that define SCD3 changes (Type 3 tracked fields)
SCD3_COLUMNS = ["first_name","last_name"]

# All columns
BUSINESS_COLS = [PK_COL] + SCD2_COLUMNS + SCD3_COLUMNS

# Column in bronze that orders the latest record per PK (choose one)
EVENT_TS_COL = "ingesttime"

# Optional "soft delete" grace policy (days). If >0, rows not seen in > grace days are closed.
SOFT_DELETE_GRACE_DAYS = None

# Far-future date for open SCD2 records
FUTURE_DATE = "9999-12-31"

# Stream checkpoint location (DBFS/ABFSS). Change to your durable storage.
CHECKPOINT_PATH = "/Volumes/labuser11612924_1758377596/_ops/scd2"

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.{DIM_TBL} (
  customer_skey BIGINT GENERATED ALWAYS AS IDENTITY,
  {PK_COL}       STRING,
  first_name      STRING,
  last_name       STRING,
  email          STRING,
  city           STRING,
  state          STRING,
  effective_date DATE,
  end_date       DATE,
  active_flag    BOOLEAN,
  last_seen_date DATE
) USING DELTA
TBLPROPERTIES (
  delta.autoOptimize.optimizeWrite = true,
  delta.autoOptimize.autoCompact   = true
)
""")

In [0]:
def normalized_sha2_256(df, cols):
    """
    Build a stable, normalized hash over 'cols':
    - trim, upper, cast to string
    - replace NULL with sentinel
    - use a non-empty separator to preserve boundaries
    """
    norm = [F.coalesce(F.trim(F.upper(F.col(c).cast("string"))), F.lit("__NULL__")) for c in cols]
    return df.withColumn("hash_256", F.sha2(F.concat_ws("||", *norm), 256))

In [0]:
def latest_per_pk(df, pk_col: str, order_ts_col: str):
    """
    Deduplicate to the latest record per PK based on order_ts_col.
    Requires order_ts_col to be a timestamp (casts if needed).
    """
    # ensure timestamp type for ordering
    if dict(df.dtypes)[order_ts_col] != "timestamp":
        df = df.withColumn(order_ts_col, F.col(order_ts_col).cast(TimestampType()))
    w = Window.partitionBy(pk_col).orderBy(F.col(order_ts_col).desc(), F.monotonically_increasing_id().desc())
    return (df
            .withColumn("_rn", F.row_number().over(w))
            .filter(F.col("_rn") == 1)
            .drop("_rn"))

In [0]:
def scd2_merge_microbatch(microbatch_df, batch_id: int):
    # --- guard ---
    try:
        is_empty = microbatch_df.isEmpty()
    except:
        is_empty = microbatch_df.rdd.isEmpty()
    if is_empty:
        if isinstance(SOFT_DELETE_GRACE_DAYS, int) and SOFT_DELETE_GRACE_DAYS > 0:
            spark.sql(f"""
              UPDATE {CATALOG}.{SCHEMA}.{DIM_TBL}
              SET end_date = current_date(), active_flag = false
              WHERE active_flag = true
                AND last_seen_date IS NOT NULL
                AND datediff(current_date(), last_seen_date) > {SOFT_DELETE_GRACE_DAYS}
            """)
        return

    # --- select needed columns ---
    selected_cols = [F.col(c) if c in microbatch_df.columns else F.lit(None).alias(c)
                     for c in (BUSINESS_COLS + [EVENT_TS_COL])]
    src = microbatch_df.select(*selected_cols)

    # --- latest per PK ---
    if dict(src.dtypes)[EVENT_TS_COL] != "timestamp":
        src = src.withColumn(EVENT_TS_COL, F.col(EVENT_TS_COL).cast(TimestampType()))
    w = Window.partitionBy(PK_COL).orderBy(F.col(EVENT_TS_COL).desc(), F.monotonically_increasing_id().desc())
    src_latest = src.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn")==1).drop("_rn")

    # --- normalized hash on SCD2 cols ---
    norm = [F.coalesce(F.trim(F.upper(F.col(c).cast("string"))), F.lit("__NULL__")) for c in SCD2_COLUMNS]
    src_h = src_latest.withColumn("hash_256", F.sha2(F.concat_ws("||", *norm), 256))

    tgt_active = spark.table(f"{CATALOG}.{SCHEMA}.{DIM_TBL}").where(F.col("active_flag")==True)
    tgt_h = tgt_active.withColumn("hash_256",
                                  F.sha2(F.concat_ws("||", *[F.coalesce(F.trim(F.upper(F.col(c).cast("string"))),
                                                                         F.lit("__NULL__"))
                                                              for c in SCD2_COLUMNS]), 256)
                                 ) \
                      .select(PK_COL, "hash_256") \
                      .withColumnRenamed("hash_256", "t_hash")

    staged = (src_h.alias("s")
              .join(tgt_h.alias("t"), on=PK_COL, how="left")
              .select(
                  *[F.col(f"s.{c}").alias(c) for c in BUSINESS_COLS],
                  F.current_date().cast(DateType()).alias("as_of_date"),
                  (F.col("t.t_hash").isNull()).alias("is_new"),
                  (F.col("t.t_hash").isNotNull() & (F.col("s.hash_256") != F.col("t.t_hash"))).alias("is_changed"),
                  (F.col("t.t_hash").isNotNull() & (F.col("s.hash_256") == F.col("t.t_hash"))).alias("is_nochange"),
              ))

    delta_tbl = DeltaTable.forName(spark, f"{CATALOG}.{SCHEMA}.{DIM_TBL}")

    # ---------- MERGE #1: close changed + refresh nochange ----------
    using_update = staged.select(*BUSINESS_COLS, "as_of_date", "is_changed", "is_nochange")
    (delta_tbl.alias("T")
      .merge(using_update.alias("S"), f"T.{PK_COL} = S.{PK_COL} AND T.active_flag = true")
      .whenMatchedUpdate(
          condition="S.is_changed",
          set={"end_date": "S.as_of_date", "active_flag": "false", "last_seen_date": "S.as_of_date"}
      )
      .whenMatchedUpdate(
          condition="S.is_nochange",
          set={"last_seen_date": "S.as_of_date"}
      )
      .execute()
    )

    # ---------- MERGE #2: insert NEW + CHANGED as a new active version ----------
    using_insert = staged.select(*BUSINESS_COLS, "as_of_date", "is_new", "is_changed")
    # after MERGE #1, there is no active row for changed keys; NOT MATCHED now fires
    inserter = (delta_tbl.alias("T")
                .merge(using_insert.alias("S"), f"T.{PK_COL} = S.{PK_COL} AND T.active_flag = true")
                .whenNotMatchedInsert(
                    condition="S.is_new OR S.is_changed",
                    values={
                        **{c: f"S.{c}" for c in BUSINESS_COLS},
                        "effective_date": "S.as_of_date",
                        "end_date":       f"DATE '{FUTURE_DATE}'",
                        "active_flag":    "true",
                        "last_seen_date": "S.as_of_date"
                    }
                ))
    inserter.execute()

    # ---------- optional soft-delete ----------
    if isinstance(SOFT_DELETE_GRACE_DAYS, int) and SOFT_DELETE_GRACE_DAYS > 0:
        spark.sql(f"""
          UPDATE {CATALOG}.{SCHEMA}.{DIM_TBL}
          SET end_date = current_date(), active_flag = false
          WHERE active_flag = true
            AND last_seen_date IS NOT NULL
            AND datediff(current_date(), last_seen_date) > {SOFT_DELETE_GRACE_DAYS}
        """)

In [0]:
# =========================================
# ====== STREAM FROM BRONZE AS SOURCE =====
# =========================================

# Read the Bronze table as a stream. This assumes Bronze is append-only.
src_stream = (spark.readStream
                   .table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TBL}")
              )

# Optional watermark for state cleanup in upstream maps (not strictly required for foreachBatch)
# Keep if you do any aggregations before foreachBatch. Here we pass through, so it's harmless.
if EVENT_TS_COL in src_stream.columns:
    src_stream = src_stream.withWatermark(EVENT_TS_COL, "7 days")

# Kick off the writer with foreachBatch
# We do no file output — the work happens inside MERGE. Use a dummy sink (e.g., "noop") or memory.
# In Databricks, use .foreachBatch and "delta" trigger; output mode "append" is irrelevant here.
query = (src_stream
         .writeStream
         .option("checkpointLocation", CHECKPOINT_PATH)
         .trigger(availableNow=True)   # or "availableNow" for one-shot; tweak to your needs
         .foreachBatch(scd2_merge_microbatch)
         .start()
        )

# If you want to block (notebooks often don't), uncomment:
# query.awaitTermination()